# Stage 2 · Smaller models for smaller GPUs: CircuitKIT circuit-aware quantization

**Tiny Aya SMS Extract** · Anshul J Amin · Lexsi AI Safety & Alignment Hackathon, Track 1

Bank SMS are private, so the extractor should run on small, local hardware. This notebook builds versions of the fine-tuned model for 16 / 8 / 4 / 2 GB GPUs and measures what each loses.

1. **Merge** the LoRA adapter into Tiny Aya Fire (fp16).
2. **CircuitKIT** finds the circuit behind the model's hardest decision, *debit vs credit*: contrastive pairs are the same SMS with "debited" ↔ "credited", scored at the `"type"` token. Node-level EAP-IG gives an importance score per attention head and MLP; we sum them per layer.
3. **Quantize** with and without protection: the most important layers stay at higher precision, the rest are compressed. A *random-protection* control (same number of layers, chosen at random) shows whether the circuit is what matters.
4. **Evaluate** every version with the same scorer on unseen formats and the hand-written set, and record memory and speed.

GPU versions use bitsandbytes (fp16, int8, nf4); the 2 GB tier uses GGUF (llama.cpp), whose quantization also compresses the large multilingual embedding table.

**How to run (Kaggle):** *File → Import Notebook* → this file. **GPU T4 x1**, **Internet ON**, secret `HF_TOKEN` (write). Run `QUICK_MODE = True` first, then `QUICK_MODE = False`.

In [ ]:
import os, sys, json, time, random, subprocess, gc, glob, shutil, traceback, re, math
from collections import Counter

QUICK_MODE = True
SEED = 7
BASE_MODEL   = "CohereLabs/tiny-aya-fire"
ADAPTER_REPO = "Meowniece/tiny-aya-sms-extract-fire-lora"      # switch to the -fire-v3-lora repo if v3 wins (decision rule)
DATA_REPO    = "Meowniece/tiny-aya-sms-extract-v3-data"
QUANT_REPO   = "Meowniece/tiny-aya-sms-extract-quantized"
MODEL_COLLECTION = "Meowniece/tiny-aya-sms-extraction-models-6af9a1a0ced10d8fb22b4ed6"
PUSH_TO_HUB  = True

N_PAIRS     = 64      # contrastive pairs for circuit discovery
N_PROTECT   = 4       # layers kept at higher precision (of 36)
RUN_CIRCUITKIT = True
RUN_GGUF    = True    # llama.cpp build + GGUF tiers (CPU evaluation)
GGUF_EVAL_N = 44      # SMS per GGUF version (CPU is slow); the hand-written set by default
EVAL_SPLITS = ["test_ood", "challenge", "test_ood_rephrased", "challenge_rephrased"]
GEN_BATCH, MAX_NEW = 16, 200

if QUICK_MODE:
    N_PAIRS, GGUF_EVAL_N = 8, 4
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("work_quant")
for d in ["src", "data", "results", "models", "figures"]:
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
random.seed(SEED)
print("work dir:", WORK, "| quick mode:", QUICK_MODE)

## 1. Install

In [ ]:
def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    print(("OK   " if r.returncode == 0 else "FAIL ") + " ".join(args))
    if r.returncode: print(r.stderr[-1500:])
    return r.returncode == 0

def secret(name):
    if os.environ.get(name): return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None
if secret("HF_TOKEN"): os.environ["HF_TOKEN"] = secret("HF_TOKEN")
print("HF_TOKEN", "found" if os.environ.get("HF_TOKEN") else "MISSING")
OK_CK = pip("circuitkit")
pip("bitsandbytes", "peft", "accelerate", "huggingface_hub", "matplotlib", "sentencepiece", "protobuf", "requests")
# check versions in a separate process so this kernel imports transformers only once
r = subprocess.run([sys.executable, "-c", "import transformers, torch, peft; print(transformers.__version__, torch.__version__, peft.__version__)"],
                   capture_output=True, text=True)
print("transformers / torch / peft:", (r.stdout or r.stderr).strip()[-300:])

## 2. Project source files

In [ ]:
# ---- src/smsgen.py ----
CONTENT = '"""Synthetic Indian bank / UPI SMS generator with exact gold labels.\n\nDesign\n------\n* Every template is a list of SEGMENTS joined by \'‖\'. The first segment is the core\n  (always kept); later segments can be dropped (optional \'?\' or by truncation). A field\n  is in the gold label ONLY if its slot survived -> the gold never contains information\n  that is not in the text (this is what lets us measure hallucination honestly).\n* \'{@inst}\' reveals the instrument without printing it (e.g. "UPI Ref" text implies UPI).\n* The LAST template of each (family, language) is held out -> `test_ood` uses message\n  formats never seen in training.\n* Bank names are FICTIONAL and messages contain no links or phone numbers, so the data\n  cannot be confused with, or reused as, real bank messages.\n"""\nfrom __future__ import annotations\n\nimport json\nimport random\nimport re\nfrom dataclasses import dataclass, field\nfrom datetime import date, timedelta\n\nFIELDS = ["is_transaction", "type", "amount", "currency", "account_last4", "instrument",\n          "counterparty", "date", "reference", "balance", "status"]\nINSTRUMENTS = ["UPI", "CARD", "NEFT", "IMPS", "RTGS", "ATM", "NACH", "AUTOPAY"]\nLANGS = ["en", "hi", "hinglish"]\n\nFICTIONAL_BANKS = ["Kestrel Bank", "Banyan Bank", "Peacock Bank", "Monsoon Bank",\n                   "Neem Bank", "Saffron Bank", "Lotus Demo Bank", "Indigo Coop Bank"]\nVPA_HANDLES = ["@kestrel", "@banyan", "@peacockpay", "@monsoon", "@neemupi", "@saffron"]\n\nMERCHANTS = ["SWIGGY", "ZOMATO", "AMAZON", "FLIPKART", "BIGBASKET", "IRCTC", "UBER", "OLA",\n             "MYNTRA", "DMART", "RELIANCE FRESH", "APOLLO PHARMACY", "BOOKMYSHOW",\n             "SHARMA GENERAL STORE", "GUPTA SWEETS", "CAFE COFFEE CORNER", "METRO CASH CARRY",\n             "HP PETROL PUMP", "DECATHLON", "NYKAA", "CROMA", "BLINKIT", "ZEPTO"]\nPERSONS = ["RAMESH KUMAR", "PRIYA SHARMA", "ANIL VERMA", "SUNITA DEVI", "ARJUN NAIR",\n           "FATIMA SHAIKH", "GURPREET SINGH", "MEERA IYER", "ROHIT DAS", "KAVYA REDDY",\n           "IMRAN KHAN", "POOJA PATEL", "VIKRAM JOSHI", "ANJALI MISHRA", "SURESH YADAV"]\nEMPLOYERS = ["ACME TECHNOLOGIES PVT LTD", "SUNRISE SOFTWARE LLP", "GANGA TEXTILES LTD",\n             "NORTHSTAR LOGISTICS", "BLUEWAVE SERVICES PVT LTD", "VEDA HOSPITALS"]\nSUBSCRIPTIONS = ["NETFLIX", "SPOTIFY", "YOUTUBE PREMIUM", "JIO FIBER", "AIRTEL POSTPAID",\n                 "TATA PLAY", "GOOGLE ONE", "HOTSTAR", "LIC PREMIUM", "SIP MUTUAL FUND"]\nBILLERS = ["CITY POWER LTD", "JAL BOARD", "METRO GAS", "AIRTEL POSTPAID", "JIO FIBER",\n           "MUNICIPAL TAX"]\nFOREIGN = [("USD", "$"), ("EUR", "€"), ("GBP", "£"), ("AED", "AED ")]\nMONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]\n\n# ---------------------------------------------------------------------------\n# Templates: family -> {"meta": {...}, "en": [...], "hi": [...], "hinglish": [...]}\n# meta: is_txn, type, status (per-template override via tuple (text, status)),\n#       inst (choices for {inst}/{@inst}), cp (counterparty pool name)\n# ---------------------------------------------------------------------------\nT = {\n    "upi_debit": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["UPI"], cp="upi_party"),\n        "en": [\n            "Your A/c {acct} is debited for {amt}‖ on {date}‖ towards {cp}‖ via {inst}‖. UPI Ref No {ref}{@inst}‖. Avl Bal {bal}",\n            "{bank}: {amt} debited from a/c {acct}‖ on {date}‖ to VPA {cp}{@inst}‖ (UPI Ref {ref}{@inst})‖. Avl bal {bal}",\n            "Sent {amt} from {bank} A/c {acct} to {cp} on {date}‖. UPI Ref: {ref}{@inst}‖. Bal: {bal}",\n        ],\n        "hi": [\n            "आपके खाते {acct} से {amt} डेबिट किए गए‖ दिनांक {date}‖, {cp} को‖ {inst} द्वारा‖। UPI संदर्भ {ref}{@inst}‖। उपलब्ध शेष {bal}",\n            "{bank}: खाता {acct} से {amt} {cp} को UPI से भेजे गए{@inst}‖ ({date})‖। संदर्भ संख्या {ref}‖। शेष राशि {bal}",\n            "{amt} का UPI भुगतान{@inst} {cp} को सफल रहा, खाता {acct}‖, दिनांक {date}‖। रेफ़ {ref}",\n        ],\n        "hinglish": [\n            "Aapke A/c {acct} se {amt} debit hue‖ {date} ko‖ {cp} ko‖ {inst} se‖. UPI Ref {ref}{@inst}‖. Avl bal {bal}",\n            "{bank}: A/c {acct} se {amt} {cp} ko UPI se bheje gaye{@inst}‖ ({date})‖. Ref no {ref}‖. Bal {bal}",\n            "{amt} ka UPI payment{@inst} {cp} ko successful, A/c {acct}‖, date {date}‖. Ref {ref}",\n        ],\n    },\n    "upi_credit": {\n        "meta": dict(is_txn=True, type="credit", status="success", inst=["UPI"], cp="upi_party"),\n        "en": [\n            "Dear Customer, your A/c {acct} is credited with {amt}‖ on {date}‖ by {cp}‖ via {inst}‖. Ref {ref}‖. Avl Bal {bal}",\n            "{amt} received in {bank} a/c {acct} from {cp}‖ on {date}‖. UPI Ref:{ref}{@inst}",\n            "Money received! {cp} sent you {amt} via {inst}‖ on {date}‖. Credited to A/c {acct}‖. Bal {bal}",\n        ],\n        "hi": [\n            "आपके खाते {acct} में {amt} जमा किए गए‖ दिनांक {date}‖, {cp} द्वारा‖ {inst} से‖। संदर्भ {ref}‖। उपलब्ध शेष {bal}",\n            "{cp} ने आपको {amt} भेजे‖, खाता {acct} में जमा‖ ({date})‖। UPI रेफ़ {ref}{@inst}",\n            "खाता {acct} में {cp} से {amt} प्राप्त हुए‖ {date} को‖ {inst} द्वारा‖। शेष {bal}",\n        ],\n        "hinglish": [\n            "Aapke A/c {acct} mein {amt} credit hue‖ {date} ko‖ {cp} se‖ {inst} dwara‖. Ref {ref}‖. Avl bal {bal}",\n            "{cp} ne aapko {amt} bheje‖, A/c {acct} mein jama‖ ({date})‖. UPI Ref {ref}{@inst}",\n            "A/c {acct} mein {cp} se {amt} mile‖ {date} ko‖ {inst} se‖. Bal {bal}",\n        ],\n    },\n    "card_spend": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["CARD"], cp="merchant", foreign=0.25),\n        "en": [\n            "You have spent {amt} on your {bank} Credit Card {card}‖ at {cp}‖ on {date}‖. Avl limit: {limit}",\n            "Txn of {amt} on Card {card} at {cp}‖ on {date}‖. Ref {ref}‖. Available credit limit {limit}",\n            "{bank} Debit Card {card} used for {amt} at {cp}‖ on {date} {time}‖. Avl bal {bal}",\n        ],\n        "hi": [\n            "आपके {bank} क्रेडिट कार्ड {card} से {cp} पर {amt} खर्च किए गए‖ दिनांक {date}‖। उपलब्ध लिमिट {limit}",\n            "कार्ड {card} से {amt} का लेनदेन {cp} पर‖, {date}‖। संदर्भ {ref}‖। उपलब्ध क्रेडिट लिमिट {limit}",\n            "{bank} डेबिट कार्ड {card} का उपयोग {cp} पर {amt} के लिए किया गया‖, {date} {time}‖। शेष {bal}",\n        ],\n        "hinglish": [\n            "Aapke {bank} Credit Card {card} se {cp} par {amt} kharch hue‖ {date} ko‖. Avl limit {limit}",\n            "Card {card} se {amt} ka txn {cp} par‖, {date}‖. Ref {ref}‖. Avl credit limit {limit}",\n            "{bank} Debit Card {card} use hua {cp} par {amt} ke liye‖, {date} {time}‖. Bal {bal}",\n        ],\n    },\n    "atm_withdrawal": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["ATM"], cp=None),\n        "en": [\n            "{amt} withdrawn from A/c {acct}‖ at {inst}‖ on {date}‖. Avl Bal {bal}",\n            "Cash withdrawal{@inst} of {amt} from your {bank} A/c {acct}‖ on {date} {time}‖. Bal: {bal}",\n            "ATM txn{@inst}: {amt} debited from a/c {acct}‖ on {date}‖. Avl bal {bal}",\n        ],\n        "hi": [\n            "खाता {acct} से {amt} की नकद निकासी{@inst}‖ {date} को‖। उपलब्ध शेष {bal}",\n            "{inst} से {amt} निकाले गए, खाता {acct}‖, दिनांक {date}‖। शेष राशि {bal}",\n            "{bank} ATM लेनदेन{@inst}: खाता {acct} से {amt} डेबिट‖, {date} {time}‖। शेष {bal}",\n        ],\n        "hinglish": [\n            "A/c {acct} se {amt} cash nikale gaye{@inst}‖ {date} ko‖. Avl bal {bal}",\n            "{inst} se {amt} withdraw hue, A/c {acct}‖, date {date}‖. Bal {bal}",\n            "{bank} ATM txn{@inst}: A/c {acct} se {amt} debit‖, {date} {time}‖. Bal {bal}",\n        ],\n    },\n    "transfer_credit": {\n        "meta": dict(is_txn=True, type="credit", status="success", inst=["NEFT", "IMPS", "RTGS"], cp="transfer_party"),\n        "en": [\n            "Your A/c {acct} has been credited with {amt}‖ on {date}‖ by {inst}‖ from {cp}‖. UTR {ref}‖. Avl Bal {bal}",\n            "{inst} credit of {amt} received in A/c {acct} from {cp}‖ on {date}‖. Ref {ref}",\n            "Salary credited: {amt} to A/c {acct}‖ from {cp}‖ via {inst}‖ on {date}‖. Bal {bal}",\n        ],\n        "hi": [\n            "आपके खाते {acct} में {inst} द्वारा {amt} जमा हुए‖, {cp} से‖, दिनांक {date}‖। UTR {ref}‖। उपलब्ध शेष {bal}",\n            "{cp} से {amt} का {inst} क्रेडिट खाता {acct} में प्राप्त‖ ({date})‖। संदर्भ {ref}",\n            "वेतन जमा: खाता {acct} में {amt}‖, {cp} द्वारा‖ {inst} से‖, {date}‖। शेष {bal}",\n        ],\n        "hinglish": [\n            "Aapke A/c {acct} mein {inst} se {amt} credit hue‖ {cp} se‖, {date} ko‖. UTR {ref}‖. Avl bal {bal}",\n            "{cp} se {amt} ka {inst} credit A/c {acct} mein aaya‖ ({date})‖. Ref {ref}",\n            "Salary aa gayi: A/c {acct} mein {amt}‖ {cp} se‖ {inst} dwara‖, {date}‖. Bal {bal}",\n        ],\n    },\n    "transfer_debit": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["NEFT", "IMPS", "RTGS"], cp="person"),\n        "en": [\n            "{amt} debited from A/c {acct}‖ on {date}‖ for {inst} transfer‖ to {cp}‖. Ref {ref}‖. Avl Bal {bal}",\n            "Your {inst} of {amt} to {cp} from A/c {acct} is successful‖. Ref No {ref}‖. Date {date}",\n            "A/c {acct} debited by {amt}‖ on {date}‖; {inst} to {cp}‖, UTR {ref}",\n        ],\n        "hi": [\n            "खाता {acct} से {amt} डेबिट‖, {date}‖, {inst} ट्रांसफ़र‖ {cp} को‖। संदर्भ {ref}‖। उपलब्ध शेष {bal}",\n            "{cp} को {amt} का आपका {inst} ट्रांसफ़र सफल रहा, खाता {acct}‖। संदर्भ संख्या {ref}‖। दिनांक {date}",\n            "खाता {acct} से {amt} कटे‖ {date} को‖; {cp} को {inst}‖, UTR {ref}",\n        ],\n        "hinglish": [\n            "A/c {acct} se {amt} debit‖, {date}‖, {inst} transfer‖ {cp} ko‖. Ref {ref}‖. Avl bal {bal}",\n            "{cp} ko {amt} ka aapka {inst} transfer successful, A/c {acct}‖. Ref no {ref}‖. Date {date}",\n            "A/c {acct} se {amt} kat gaye‖ {date} ko‖; {cp} ko {inst}‖, UTR {ref}",\n        ],\n    },\n    "refund_reversal": {\n        "meta": dict(is_txn=True, type="credit", status="success", inst=[], cp="merchant"),\n        "en": [\n            ("Refund of {amt} from {cp} credited to your A/c {acct}‖ on {date}‖. Avl Bal {bal}", "success"),\n            ("{amt} debited on {date} for txn to {cp} has been reversed to your A/c {acct}‖. Ref {ref}", "reversed"),\n            ("Your refund of {amt} for order at {cp} is processed to Card {card}‖ on {date}", "success"),\n        ],\n        "hi": [\n            ("{cp} से {amt} का रिफ़ंड आपके खाते {acct} में जमा किया गया‖, दिनांक {date}‖। उपलब्ध शेष {bal}", "success"),\n            ("{date} को {cp} के लेनदेन में कटे {amt} आपके खाते {acct} में वापस कर दिए गए हैं‖। संदर्भ {ref}", "reversed"),\n            ("{cp} पर ऑर्डर का {amt} रिफ़ंड कार्ड {card} में भेज दिया गया है‖, {date}", "success"),\n        ],\n        "hinglish": [\n            ("{cp} se {amt} ka refund aapke A/c {acct} mein credit hua‖ {date} ko‖. Avl bal {bal}", "success"),\n            ("{date} ko {cp} txn mein kate {amt} aapke A/c {acct} mein wapas aa gaye‖. Ref {ref}", "reversed"),\n            ("{cp} order ka {amt} refund Card {card} mein bhej diya gaya‖, {date}", "success"),\n        ],\n    },\n    "failed_txn": {\n        "meta": dict(is_txn=True, type="debit", status="failed", inst=["UPI"], cp="merchant"),\n        "en": [\n            "Txn of {amt} to {cp} via {inst} FAILED‖ on {date}‖. Amount, if debited, will be refunded‖. Ref {ref}",\n            "Your payment of {amt} to {cp} could not be processed‖ (UPI Ref {ref}{@inst})‖. A/c {acct} not debited",\n            "Transaction declined: {amt} at {cp} on Card {card}‖ on {date}‖. Reason: insufficient balance",\n        ],\n        "hi": [\n            "{inst} द्वारा {cp} को {amt} का लेनदेन विफल रहा‖, दिनांक {date}‖। अगर राशि कटी है तो वापस आ जाएगी‖। संदर्भ {ref}",\n            "{cp} को {amt} का आपका भुगतान नहीं हो सका‖ (UPI रेफ़ {ref}{@inst})‖। खाता {acct} से राशि नहीं कटी",\n            "लेनदेन अस्वीकृत: कार्ड {card} से {cp} पर {amt}‖, {date}‖। कारण: अपर्याप्त शेष",\n        ],\n        "hinglish": [\n            "{inst} se {cp} ko {amt} ka txn fail ho gaya‖ {date} ko‖. Paisa kata hai to wapas aa jayega‖. Ref {ref}",\n            "{cp} ko {amt} ka aapka payment nahi ho paya‖ (UPI Ref {ref}{@inst})‖. A/c {acct} se paisa nahi kata",\n            "Txn decline hua: Card {card} se {cp} par {amt}‖, {date}‖. Reason: balance kam hai",\n        ],\n    },\n    "autopay_emi": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["NACH", "AUTOPAY"], cp="subscription"),\n        "en": [\n            "{amt} debited from A/c {acct} towards EMI for loan {ref}‖ via {inst}‖ on {date}‖. Avl Bal {bal}",\n            "AutoPay{@inst=AUTOPAY} of {amt} for {cp} executed from A/c {acct}‖ on {date}‖. Ref {ref}",\n            "Your {cp} subscription of {amt} was paid via {inst}‖ from A/c {acct}‖ on {date}",\n        ],\n        "hi": [\n            "लोन {ref} की EMI के लिए खाता {acct} से {amt} डेबिट‖ {inst} द्वारा‖, दिनांक {date}‖। उपलब्ध शेष {bal}",\n            "{cp} के लिए {amt} का ऑटोपे{@inst=AUTOPAY} खाता {acct} से पूरा हुआ‖, {date}‖। संदर्भ {ref}",\n            "आपकी {cp} सदस्यता के {amt} {inst} से चुकाए गए‖, खाता {acct}‖, {date}",\n        ],\n        "hinglish": [\n            "Loan {ref} ki EMI ke liye A/c {acct} se {amt} debit‖ {inst} se‖, {date} ko‖. Avl bal {bal}",\n            "{cp} ke liye {amt} ka AutoPay{@inst=AUTOPAY} A/c {acct} se ho gaya‖, {date}‖. Ref {ref}",\n            "Aapki {cp} subscription ke {amt} {inst} se pay hue‖, A/c {acct}‖, {date}",\n        ],\n    },\n    # ---------------- hard negatives: money mentioned, but nothing moved ----------------\n    "otp": {\n        "meta": dict(is_txn=False, cp="merchant"),\n        "en": [\n            "{otp} is the OTP for your txn of {amt} at {cp} on {bank} Card {card}. Valid for 5 mins. Do not share it with anyone.",\n            "OTP {otp} to authorise payment of {amt} to {cp}. Never share your OTP. -{bank}",\n        ],\n        "hi": [\n            "{otp} आपके {bank} कार्ड {card} से {cp} पर {amt} के लेनदेन का OTP है। 5 मिनट तक मान्य। इसे किसी से साझा न करें।",\n            "{cp} को {amt} के भुगतान के लिए OTP {otp} है। अपना OTP कभी किसी को न बताएँ। -{bank}",\n        ],\n        "hinglish": [\n            "{otp} aapke {bank} Card {card} se {cp} par {amt} ke txn ka OTP hai. 5 min tak valid. Kisi ke saath share na karein.",\n            "{cp} ko {amt} payment ke liye OTP {otp} hai. Apna OTP kabhi kisi ko na batayein. -{bank}",\n        ],\n    },\n    "promo": {\n        "meta": dict(is_txn=False, cp=None),\n        "en": [\n            "Get cashback up to {amt} on your next UPI payment with {bank}! Offer valid till {date}. T&C apply.",\n            "Pre-approved loan of up to {amt} is waiting for you. Apply in the {bank} app today.",\n        ],\n        "hi": [\n            "{bank} के साथ अगले UPI भुगतान पर {amt} तक का कैशबैक पाएँ! ऑफ़र {date} तक मान्य। नियम व शर्तें लागू।",\n            "आपके लिए {amt} तक का प्री-अप्रूव्ड लोन तैयार है। आज ही {bank} ऐप में आवेदन करें।",\n        ],\n        "hinglish": [\n            "{bank} ke saath agle UPI payment par {amt} tak cashback paayein! Offer {date} tak valid. T&C apply.",\n            "Aapke liye {amt} tak ka pre-approved loan ready hai. Aaj hi {bank} app mein apply karein.",\n        ],\n    },\n    "bill_due": {\n        "meta": dict(is_txn=False, cp="biller"),\n        "en": [\n            "Your {bank} Credit Card {card} bill of {amt} is due on {date}. Minimum due {amt2}. Please pay on time to avoid charges.",\n            "Reminder: {cp} bill of {amt} is due on {date}. Ignore if already paid.",\n        ],\n        "hi": [\n            "आपके {bank} क्रेडिट कार्ड {card} का {amt} का बिल {date} को देय है। न्यूनतम देय राशि {amt2}। शुल्क से बचने के लिए समय पर भुगतान करें।",\n            "रिमाइंडर: {cp} का {amt} का बिल {date} को देय है। यदि भुगतान कर दिया है तो इसे अनदेखा करें।",\n        ],\n        "hinglish": [\n            "Aapke {bank} Credit Card {card} ka {amt} ka bill {date} ko due hai. Minimum due {amt2}. Charges se bachne ke liye time par pay karein.",\n            "Reminder: {cp} ka {amt} ka bill {date} ko due hai. Pay kar diya hai to ignore karein.",\n        ],\n    },\n    "scheduled_debit": {\n        "meta": dict(is_txn=False, cp="subscription"),\n        "en": [\n            "{amt} will be debited from A/c {acct} on {date} towards {cp} AutoPay. Please maintain sufficient balance.",\n            "Upcoming EMI of {amt} for loan {ref} will be deducted on {date}. -{bank}",\n        ],\n        "hi": [\n            "{date} को {cp} ऑटोपे के लिए खाता {acct} से {amt} डेबिट किए जाएँगे। कृपया पर्याप्त शेष रखें।",\n            "लोन {ref} की {amt} की आगामी EMI {date} को कटेगी। -{bank}",\n        ],\n        "hinglish": [\n            "{date} ko {cp} AutoPay ke liye A/c {acct} se {amt} debit honge. Kripya sufficient balance rakhein.",\n            "Loan {ref} ki {amt} ki agli EMI {date} ko kategi. -{bank}",\n        ],\n    },\n    "balance_info": {\n        "meta": dict(is_txn=False, cp=None),\n        "en": [\n            "Avl bal in your {bank} A/c {acct} is {bal} as on {date}.",\n            "Dear Customer, the balance in A/c {acct} on {date} is {bal}. -{bank}",\n        ],\n        "hi": [\n            "{date} को आपके {bank} खाते {acct} में उपलब्ध शेष {bal} है।",\n            "प्रिय ग्राहक, खाता {acct} में {date} को शेष राशि {bal} है। -{bank}",\n        ],\n        "hinglish": [\n            "{date} ko aapke {bank} A/c {acct} mein avl bal {bal} hai.",\n            "Dear Customer, A/c {acct} mein {date} ko balance {bal} hai. -{bank}",\n        ],\n    },\n}\nTXN_FAMILIES = [f for f, v in T.items() if v["meta"]["is_txn"]]\nNEG_FAMILIES = [f for f, v in T.items() if not v["meta"]["is_txn"]]\n\n\n# ---------------------------------------------------------------------------\n# Value renderers\n# ---------------------------------------------------------------------------\ndef indian_group(n: int) -> str:\n    s = str(n)\n    if len(s) <= 3:\n        return s\n    head, tail = s[:-3], s[-3:]\n    parts = []\n    while len(head) > 2:\n        parts.insert(0, head[-2:])\n        head = head[:-2]\n    if head:\n        parts.insert(0, head)\n    return ",".join(parts) + "," + tail\n\n\ndef fmt_number(v: float, rng: random.Random) -> str:\n    whole = int(round(v * 100)) // 100\n    paise = int(round(v * 100)) % 100\n    style = rng.choice(["plain", "indian", "indian", "western"])\n    w = str(whole) if style == "plain" else indian_group(whole) if style == "indian" else f"{whole:,}"\n    if paise == 0 and rng.random() < 0.5:\n        return w\n    return f"{w}.{paise:02d}"\n\n\nINR_FORMATS = {\n    "en": ["Rs.{n}", "Rs {n}", "INR {n}", "₹{n}", "Rs.{n}/-", "INR{n}"],\n    "hi": ["₹{n}", "रु. {n}", "रुपये {n}", "Rs.{n}", "₹ {n}"],\n    "hinglish": ["Rs {n}", "Rs.{n}", "₹{n}", "{n} rupaye", "INR {n}"],\n}\n\n\ndef rand_amount(rng: random.Random, lo=1.0, hi=500000.0) -> float:\n    import math\n    v = math.exp(rng.uniform(math.log(lo), math.log(hi)))\n    return round(v, 2) if rng.random() < 0.6 else float(int(v))\n\n\ndef fmt_inr(v: float, lang: str, rng: random.Random) -> str:\n    return rng.choice(INR_FORMATS[lang]).format(n=fmt_number(v, rng))\n\n\ndef fmt_foreign(v: float, code: str, sym: str, rng: random.Random) -> str:\n    n = f"{v:.2f}"\n    return rng.choice([f"{code} {n}", f"{sym}{n}" if sym.strip() != code else f"{code} {n}", f"{code}{n}"])\n\n\ndef fmt_date(d: date, rng: random.Random) -> str:\n    style = rng.randrange(8)\n    m = MONTHS[d.month - 1]\n    return [\n        f"{d.day:02d}-{d.month:02d}-{d.year % 100:02d}",\n        f"{d.day:02d}/{d.month:02d}/{d.year}",\n        f"{d.day:02d}{m}{d.year % 100:02d}",\n        f"{d.day:02d}-{m}-{d.year}",\n        f"{d.year}-{d.month:02d}-{d.day:02d}",\n        f"{d.day:02d}.{d.month:02d}.{d.year}",\n        f"{d.day} {m} {d.year}",\n        f"{d.day:02d}-{m.upper()}-{d.year % 100:02d}",\n    ][style]\n\n\ndef fmt_acct(l4: str, lang: str, rng: random.Random) -> str:\n    opts = [f"XX{l4}", f"XXXXXX{l4}", f"*{l4}", f"x{l4}", f"XXXX{l4}", f"**{l4}"]\n    if lang == "en":\n        opts += [f"no. XX{l4}", f"ending {l4}"]\n    return rng.choice(opts)\n\n\ndef fmt_card(l4: str, rng: random.Random) -> str:\n    return rng.choice([f"XX{l4}", f"ending {l4}", f"**** {l4}", f"xx{l4}", f"XXXX-{l4}"])\n\n\ndef rand_ref(inst: str | None, rng: random.Random) -> str:\n    if inst in ("NEFT", "RTGS") and rng.random() < 0.6:\n        return rng.choice("NHSKBP") + "".join(rng.choice("0123456789") for _ in range(15))\n    if rng.random() < 0.15:\n        return "".join(rng.choice("ABCDEFGHJKLMNPQRSTUVWXYZ0123456789") for _ in range(10))\n    return "".join(rng.choice("0123456789") for _ in range(12))\n\n\ndef rand_counterparty(pool: str | None, rng: random.Random) -> str | None:\n    if pool is None:\n        return None\n    if pool == "upi_party":\n        r = rng.random()\n        if r < 0.4:\n            return rng.choice(MERCHANTS)\n        if r < 0.7:\n            return rng.choice(PERSONS)\n        name = rng.choice(PERSONS).lower().split()\n        return f"{name[0]}.{name[1][:1]}{rng.randint(1, 99)}{rng.choice(VPA_HANDLES)}"\n    if pool == "transfer_party":\n        return rng.choice(EMPLOYERS) if rng.random() < 0.5 else rng.choice(PERSONS)\n    return rng.choice({"merchant": MERCHANTS, "person": PERSONS, "subscription": SUBSCRIPTIONS,\n                       "biller": BILLERS}[pool])\n\n\n# ---------------------------------------------------------------------------\n# Perturbations (edge cases). Each returns (text, tag) and never changes gold values.\n# ---------------------------------------------------------------------------\nDEV_DIGITS = str.maketrans("0123456789", "०१२३४५६७८९")\n\n\ndef perturb(text: str, lang: str, rng: random.Random, p: float = 0.35) -> tuple[str, list[str]]:\n    tags = []\n    if rng.random() > p:\n        return text, tags\n    ops = ["spacing", "header", "promo_tail", "footer"]\n    if lang in ("en", "hinglish"):\n        ops += ["upper", "lower"]\n    if lang == "hi":\n        ops += ["dev_digits"]\n    for op in rng.sample(ops, k=rng.choice([1, 1, 2])):\n        if op == "upper":\n            text = text.upper()\n        elif op == "lower":\n            text = text.lower()\n        elif op == "spacing":\n            # remove a space between a number and a letter, or double a space\n            if rng.random() < 0.5:\n                text = re.sub(r"(?<=\\d) (?=[^\\W\\d])", "", text, count=1)\n            else:\n                text = re.sub(r" ", "  ", text, count=1)\n        elif op == "header":\n            text = rng.choice(["[Kestrel Bk] ", "BNYN: ", "Alert: ", "Txn Alert- "]) + text\n        elif op == "footer":\n            text = text + rng.choice([" -Kestrel Bank", " Not you? Report via your bank app.",\n                                      " Download our app for details.", " - Neem Bank"])\n        elif op == "promo_tail":\n            text = text + rng.choice([" Get 5% cashback up to Rs.100 on your next txn!",\n                                      " Earn 2X reward points on weekend spends.",\n                                      " Win vouchers worth Rs 500. T&C apply."])\n        elif op == "dev_digits":\n            text = text.translate(DEV_DIGITS)\n        tags.append(op)\n    return text, tags\n\n\n# ---------------------------------------------------------------------------\n# Rendering one example\n# ---------------------------------------------------------------------------\n@dataclass\nclass Example:\n    id: str\n    family: str\n    lang: str\n    template_id: str\n    sms: str\n    gold: dict\n    tags: list = field(default_factory=list)\n    split: str = ""\n\n    def to_json(self) -> dict:\n        return {"id": self.id, "family": self.family, "lang": self.lang,\n                "template_id": self.template_id, "sms": self.sms,\n                "gold": self.gold, "tags": self.tags, "split": self.split}\n\n\n_SLOT = re.compile(r"\\{(@?)(\\w+)(?:=(\\w+))?\\}")\n\n\ndef render(family: str, lang: str, t_idx: int, rng: random.Random, ex_id: str,\n           noise_p: float = 0.35, allow_drop: bool = True) -> Example:\n    spec = T[family]\n    meta = spec["meta"]\n    tmpl = spec[lang][t_idx]\n    status = meta.get("status")\n    if isinstance(tmpl, tuple):\n        tmpl, status = tmpl\n    segments = tmpl.split("‖")\n    keep = [True] * len(segments)\n    tags = []\n    if allow_drop and meta["is_txn"] and len(segments) > 1:\n        # optional-segment dropping (30%) and truncation (10%)\n        for k in range(1, len(segments)):\n            if rng.random() < 0.15:\n                keep[k] = False\n        if any(not x for x in keep[1:]):\n            tags.append("missing_fields")\n        if rng.random() < 0.10:\n            cut = rng.randint(1, len(segments) - 1)\n            for k in range(cut, len(segments)):\n                keep[k] = False\n            tags.append("truncated")\n\n    inst = rng.choice(meta["inst"]) if meta.get("inst") else None\n    foreign = meta.get("foreign", 0) and rng.random() < meta["foreign"]\n    amount = rand_amount(rng, 1, 3000) if foreign else rand_amount(rng)\n    code, sym = rng.choice(FOREIGN) if foreign else ("INR", "₹")\n    d = date(2025, 1, 1) + timedelta(days=rng.randrange(0, 730))\n    acct = f"{rng.randint(0, 9999):04d}"\n    card = f"{rng.randint(0, 9999):04d}"\n    values = {\n        "amt": fmt_foreign(amount, code, sym, rng) if foreign else fmt_inr(amount, lang, rng),\n        "amt2": fmt_inr(round(amount * 0.05, 2) if amount > 100 else amount, lang, rng),\n        "acct": fmt_acct(acct, lang, rng),\n        "card": fmt_card(card, rng),\n        "date": fmt_date(d, rng),\n        "time": f"{rng.randint(0, 23):02d}:{rng.randint(0, 59):02d}",\n        "cp": rand_counterparty(meta.get("cp"), rng) or "",\n        "ref": rand_ref(inst, rng),\n        "bal": fmt_inr(rand_amount(rng, 10, 900000), lang, rng),\n        "limit": fmt_inr(rand_amount(rng, 1000, 500000), lang, rng),\n        "inst": {"AUTOPAY": "AutoPay"}.get(inst, inst) if inst else "",\n        "bank": rng.choice(FICTIONAL_BANKS),\n        "otp": f"{rng.randint(0, 999999):06d}",\n    }\n    gold_values = {\n        "amt": ("amount", amount), "acct": ("account_last4", acct), "card": ("account_last4", card),\n        "date": ("date", d.isoformat()), "cp": ("counterparty", values["cp"] or None),\n        "ref": ("reference", values["ref"]), "bal": ("balance", None),\n        "inst": ("instrument", inst),\n    }\n    # balance gold value is the numeric value we rendered; re-derive from text below\n    bal_value = None\n\n    revealed = {}\n    parts = []\n    for seg, k in zip(segments, keep):\n        if not k:\n            continue\n\n        def sub(m):\n            hidden, name, fixed = m.group(1), m.group(2), m.group(3)\n            if name in gold_values:\n                fname, val = gold_values[name]\n                if fixed:\n                    val = fixed\n                revealed[fname] = val\n                if name == "card":\n                    revealed["instrument"] = "CARD"\n                if name == "amt":\n                    revealed["currency"] = code\n            if name == "bal":\n                revealed["balance"] = "__BAL__"\n            return "" if hidden else values[name]\n        parts.append(_SLOT.sub(sub, seg))\n    text = "".join(parts).strip()\n    # numeric balance value from rendered string\n    if "balance" in revealed:\n        bal_value = parse_rendered_number(values["bal"])\n        revealed["balance"] = bal_value\n    if text and text[-1] not in ".।!":\n        text += "." if lang != "hi" else "।"\n\n    gold = {f: None for f in FIELDS}\n    gold["is_transaction"] = bool(meta["is_txn"])\n    if meta["is_txn"]:\n        gold["type"] = meta["type"]\n        gold["status"] = status\n        for k, v in revealed.items():\n            gold[k] = v\n        if foreign:\n            tags.append("foreign_currency")\n        if "{limit}" in "".join(s for s, k in zip(segments, keep) if k):\n            tags.append("limit_distractor")\n    else:\n        tags.append("hard_negative")\n    text, ptags = perturb(text, lang, rng, p=noise_p)\n    tags += ptags\n    return Example(id=ex_id, family=family, lang=lang, template_id=f"{family}/{lang}/{t_idx}",\n                   sms=text, gold=gold, tags=tags)\n\n\n_NUM = re.compile(r"[\\d,]+(?:\\.\\d+)?")\n\n\ndef parse_rendered_number(s: str) -> float:\n    m = _NUM.search(s)\n    return float(m.group(0).replace(",", "")) if m else None\n\n\n# ---------------------------------------------------------------------------\n# Dataset builder with template-level held-out split\n# ---------------------------------------------------------------------------\ndef build_dataset(n_train=2400, n_dev=200, n_test_id=400, n_test_ood=400, seed=7,\n                  neg_share=0.3, noise_p=0.35, allow_drop=True,\n                  exclude_sms: set | None = None) -> dict[str, list[Example]]:\n    """noise_p / allow_drop / neg_share = 0 give the \'clean data\' ablation."""\n    rng = random.Random(seed)\n    seen_sms = set(exclude_sms or ())\n    out = {"train": [], "dev": [], "test_id": [], "test_ood": []}\n\n    def pick(held_out: bool):\n        fam = rng.choice(NEG_FAMILIES) if rng.random() < neg_share else rng.choice(TXN_FAMILIES)\n        lang = rng.choice(LANGS)\n        n_t = len(T[fam][lang])\n        t_idx = n_t - 1 if held_out else rng.randrange(0, n_t - 1)\n        return fam, lang, t_idx\n\n    for split, n in [("test_ood", n_test_ood), ("test_id", n_test_id), ("dev", n_dev), ("train", n_train)]:\n        i = 0\n        tries = 0\n        while len(out[split]) < n:\n            tries += 1\n            fam, lang, t_idx = pick(held_out=(split == "test_ood"))\n            ex = render(fam, lang, t_idx, rng, f"{split}-{i:05d}", noise_p=noise_p,\n                        allow_drop=allow_drop)\n            if ex.sms in seen_sms:      # exact-duplicate guard across ALL splits (no leakage)\n                continue\n            seen_sms.add(ex.sms)\n            ex.split = split\n            out[split].append(ex)\n            i += 1\n    return out\n\n\ndef gold_to_str(gold: dict) -> str:\n    return json.dumps({k: gold[k] for k in FIELDS}, ensure_ascii=False)\n\n\nSYSTEM_PROMPT = (\n    "You extract structured data from Indian bank/UPI SMS (English, Hindi or Hinglish). "\n    "Reply with ONLY one JSON object with exactly these keys: "\n    "is_transaction (true only if money actually moved or a transaction was attempted/failed/reversed; "\n    "false for OTPs, offers, reminders, future debits, balance info), "\n    "type (\\"debit\\"|\\"credit\\"|null), amount (number), currency (ISO code, e.g. \\"INR\\"), "\n    "account_last4 (last 4 digits as string), "\n    "instrument (\\"UPI\\"|\\"CARD\\"|\\"NEFT\\"|\\"IMPS\\"|\\"RTGS\\"|\\"ATM\\"|\\"NACH\\"|\\"AUTOPAY\\"|null), "\n    "counterparty (merchant/person/VPA as written), date (YYYY-MM-DD; SMS dates are day-first), "\n    "reference (string), balance (number; available balance only, never credit limit), "\n    "status (\\"success\\"|\\"failed\\"|\\"reversed\\"|null). "\n    "Use null for anything not stated in the SMS. Never guess. Use ASCII digits. "\n    "If is_transaction is false, set every other key to null."\n)\n\n\ndef build_messages(sms: str, few_shot: list[tuple[str, str]] | None = None) -> list[dict]:\n    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]\n    for s, g in (few_shot or []):\n        msgs.append({"role": "user", "content": f"SMS: {s}"})\n        msgs.append({"role": "assistant", "content": g})\n    msgs.append({"role": "user", "content": f"SMS: {sms}"})\n    return msgs\n\n\nif __name__ == "__main__":\n    ds = build_dataset()\n    for k, v in ds.items():\n        print(k, len(v))\n    for ex in ds["train"][:6]:\n        print(ex.lang, ex.family, ex.tags, "\\n ", ex.sms, "\\n ", gold_to_str(ex.gold))\n'
open(f"{WORK}/src/smsgen.py", "w", encoding="utf-8").write(CONTENT)
print('wrote src/smsgen.py', len(CONTENT), 'chars')

In [ ]:
# ---- src/scoring.py ----
CONTENT = '"""Parsing, schema validation and metrics for SMS -> JSON extraction."""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport random\nimport re\nimport unicodedata\nfrom collections import defaultdict\n\nFIELDS = ["is_transaction", "type", "amount", "currency", "account_last4", "instrument",\n          "counterparty", "date", "reference", "balance", "status"]\nVALUE_FIELDS = FIELDS[1:]\nENUMS = {\n    "type": {"debit", "credit"},\n    "instrument": {"UPI", "CARD", "NEFT", "IMPS", "RTGS", "ATM", "NACH", "AUTOPAY"},\n    "status": {"success", "failed", "reversed"},\n}\n_DEV = str.maketrans("०१२३४५६७८९", "0123456789")\n\n\n# ---------------------------------------------------------------------------\n# Parsing\n# ---------------------------------------------------------------------------\ndef extract_json(text: str) -> dict | None:\n    """Return the first JSON object in `text` (tolerates code fences / chatter)."""\n    if not text:\n        return None\n    t = text.strip()\n    t = re.sub(r"^```(?:json)?\\s*|\\s*```$", "", t)\n    start = t.find("{")\n    while start != -1:\n        depth, in_str, esc = 0, False, False\n        for i in range(start, len(t)):\n            c = t[i]\n            if in_str:\n                if esc:\n                    esc = False\n                elif c == "\\\\":\n                    esc = True\n                elif c == \'"\':\n                    in_str = False\n            elif c == \'"\':\n                in_str = True\n            elif c == "{":\n                depth += 1\n            elif c == "}":\n                depth -= 1\n                if depth == 0:\n                    try:\n                        obj = json.loads(t[start:i + 1])\n                        return obj if isinstance(obj, dict) else None\n                    except json.JSONDecodeError:\n                        break\n        start = t.find("{", start + 1)\n    return None\n\n\ndef schema_errors(obj: dict | None) -> list[str]:\n    if obj is None:\n        return ["not_json"]\n    errs = []\n    keys = set(obj)\n    if keys != set(FIELDS):\n        if set(FIELDS) - keys:\n            errs.append("missing_keys")\n        if keys - set(FIELDS):\n            errs.append("extra_keys")\n    if not isinstance(obj.get("is_transaction"), bool):\n        errs.append("bad_is_transaction")\n    for k, allowed in ENUMS.items():\n        v = obj.get(k)\n        if v is not None and (not isinstance(v, str) or v not in allowed):\n            errs.append(f"bad_{k}")\n    for k in ("amount", "balance"):\n        v = obj.get(k)\n        if v is not None and not (isinstance(v, (int, float)) and not isinstance(v, bool)):\n            errs.append(f"bad_{k}")\n    d = obj.get("date")\n    if d is not None and not (isinstance(d, str) and re.fullmatch(r"\\d{4}-\\d{2}-\\d{2}", d)):\n        errs.append("bad_date")\n    a = obj.get("account_last4")\n    if a is not None and not (isinstance(a, str) and re.fullmatch(r"\\d{4}", a)):\n        errs.append("bad_account_last4")\n    return errs\n\n\n# ---------------------------------------------------------------------------\n# Normalisation & field comparison (lenient on formatting, strict on meaning)\n# ---------------------------------------------------------------------------\ndef _num(v):\n    if v is None:\n        return None\n    if isinstance(v, bool):\n        return None\n    if isinstance(v, (int, float)):\n        return float(v)\n    s = str(v).translate(_DEV)\n    m = re.search(r"\\d[\\d,]*(?:\\.\\d+)?", s)\n    return float(m.group(0).replace(",", "")) if m else None\n\n\ndef _norm_str(v):\n    if v is None:\n        return None\n    s = unicodedata.normalize("NFC", str(v)).translate(_DEV).casefold()\n    s = re.sub(r"\\s+", " ", s).strip()\n    return s or None\n\n\ndef _norm_alnum(v):\n    s = _norm_str(v)\n    if not s:\n        return None\n    s = "".join(ch for ch in s if ch in "@." or unicodedata.category(ch)[0] in "LMN")   # keeps Devanagari letters\n    return s or None\n\n\ndef norm_field(k, v):\n    if v is None or (isinstance(v, str) and v.strip().lower() in ("", "null", "none", "n/a")):\n        return None\n    if k in ("amount", "balance"):\n        return _num(v)\n    if k == "account_last4":\n        d = re.sub(r"\\D", "", str(v).translate(_DEV))\n        return d[-4:] if d else None\n    if k in ("instrument", "currency"):\n        return _norm_str(v).upper() if _norm_str(v) else None\n    if k in ("type", "status"):\n        return _norm_str(v)\n    if k in ("reference", "counterparty"):\n        return _norm_alnum(v)\n    if k == "date":\n        return _norm_str(v)\n    if k == "is_transaction":\n        if isinstance(v, bool):\n            return v\n        return {"true": True, "false": False}.get(str(v).lower())\n    return _norm_str(v)\n\n\ndef field_equal(k, pred, gold) -> bool:\n    p, g = norm_field(k, pred), norm_field(k, gold)\n    if g is None or p is None:\n        return g is None and p is None\n    if k in ("amount", "balance"):\n        return abs(p - g) < 0.005\n    return p == g\n\n\n# ---------------------------------------------------------------------------\n# Per-example and aggregate scoring\n# ---------------------------------------------------------------------------\ndef score_example(raw_output: str, gold: dict) -> dict:\n    obj = extract_json(raw_output)\n    errs = schema_errors(obj)\n    pred = obj or {}\n    res = {"json_parsed": obj is not None, "schema_valid": obj is not None and not errs,\n           "schema_errors": errs}\n    correct = {}\n    for k in FIELDS:\n        correct[k] = field_equal(k, pred.get(k), gold.get(k))\n    res["field_correct"] = correct\n    res["exact_match"] = obj is not None and all(correct.values())\n    # hallucination: predicted a value where gold is null; omission: the reverse\n    hall = omit = gold_null = gold_val = 0\n    for k in VALUE_FIELDS:\n        g = norm_field(k, gold.get(k))\n        p = norm_field(k, pred.get(k))\n        if g is None:\n            gold_null += 1\n            hall += p is not None\n        else:\n            gold_val += 1\n            omit += p is None\n    res.update(hallucinated=hall, gold_null=gold_null, omitted=omit, gold_valued=gold_val)\n    res["pred_is_txn"] = norm_field("is_transaction", pred.get("is_transaction"))\n    res["gold_is_txn"] = gold["is_transaction"]\n    return res\n\n\ndef aggregate(scored: list[dict]) -> dict:\n    n = len(scored)\n    if n == 0:\n        return {}\n    out = {\n        "n": n,\n        "json_valid": sum(s["json_parsed"] for s in scored) / n,\n        "schema_valid": sum(s["schema_valid"] for s in scored) / n,\n        "exact_match": sum(s["exact_match"] for s in scored) / n,\n    }\n    # micro field accuracy over value fields on transaction examples\n    txn = [s for s in scored if s["gold_is_txn"]]\n    if txn:\n        out["field_acc_txn"] = sum(s["field_correct"][k] for s in txn for k in VALUE_FIELDS) / (\n            len(txn) * len(VALUE_FIELDS))\n        for k in VALUE_FIELDS:\n            out[f"acc_{k}"] = sum(s["field_correct"][k] for s in txn) / len(txn)\n    gn = sum(s["gold_null"] for s in scored)\n    gv = sum(s["gold_valued"] for s in scored)\n    out["hallucination_rate"] = sum(s["hallucinated"] for s in scored) / gn if gn else float("nan")\n    out["omission_rate"] = sum(s["omitted"] for s in scored) / gv if gv else float("nan")\n    # transaction detection (positive class = NOT a transaction, i.e. catching hard negatives)\n    tp = sum(1 for s in scored if s["gold_is_txn"] is False and s["pred_is_txn"] is False)\n    fp = sum(1 for s in scored if s["gold_is_txn"] is True and s["pred_is_txn"] is False)\n    fn = sum(1 for s in scored if s["gold_is_txn"] is False and s["pred_is_txn"] is not False)\n    prec = tp / (tp + fp) if tp + fp else float("nan")\n    rec = tp / (tp + fn) if tp + fn else float("nan")\n    out["nontxn_precision"], out["nontxn_recall"] = prec, rec\n    out["nontxn_f1"] = 2 * prec * rec / (prec + rec) if prec and rec and not math.isnan(prec + rec) else 0.0\n    out["is_txn_acc"] = sum(1 for s in scored if s["pred_is_txn"] == s["gold_is_txn"]) / n\n    return out\n\n\ndef breakdown(rows: list[dict], scored: list[dict], key: str) -> dict:\n    groups = defaultdict(list)\n    for r, s in zip(rows, scored):\n        vals = r.get(key)\n        for v in (vals if isinstance(vals, list) else [vals]):\n            groups[v].append(s)\n    return {g: aggregate(v) for g, v in sorted(groups.items(), key=lambda x: str(x[0]))}\n\n\n# ---------------------------------------------------------------------------\n# Statistics\n# ---------------------------------------------------------------------------\ndef mcnemar_exact(a: list[bool], b: list[bool]) -> dict:\n    """Paired exact McNemar test (a = system A correct, b = system B correct)."""\n    n01 = sum(1 for x, y in zip(a, b) if not x and y)\n    n10 = sum(1 for x, y in zip(a, b) if x and not y)\n    n = n01 + n10\n    if n == 0:\n        return {"b_only": n01, "a_only": n10, "p_value": 1.0}\n    k = min(n01, n10)\n    p = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n\n    return {"b_only": n01, "a_only": n10, "p_value": min(1.0, 2 * p)}\n\n\ndef bootstrap_ci(values: list[float], n_boot: int = 2000, seed: int = 0) -> tuple[float, float]:\n    rng = random.Random(seed)\n    n = len(values)\n    if n == 0:\n        return (float("nan"), float("nan"))\n    means = sorted(sum(values[rng.randrange(n)] for _ in range(n)) / n for _ in range(n_boot))\n    return means[int(0.025 * n_boot)], means[int(0.975 * n_boot) - 1]\n'
open(f"{WORK}/src/scoring.py", "w", encoding="utf-8").write(CONTENT)
print('wrote src/scoring.py', len(CONTENT), 'chars')

In [ ]:
sys.path.insert(0, f"{WORK}/src")
import importlib, smsgen, scoring as S
from smsgen import build_messages, gold_to_str, SYSTEM_PROMPT
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import hf_hub_download, HfApi, login
if os.environ.get("HF_TOKEN"): login(token=os.environ["HF_TOKEN"])
api = HfApi()
def load_jsonl(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
splits = {}
for name in ["train_v3"] + EVAL_SPLITS:
    try:
        splits[name] = load_jsonl(hf_hub_download(DATA_REPO, f"data/{name}.jsonl", repo_type="dataset"))
    except Exception as e:
        print(name, "not available:", str(e)[:80])
EVAL_SPLITS = [s for s in EVAL_SPLITS if splits.get(s)]
if QUICK_MODE:
    splits = {k: (v if k == "train_v3" else v[:6]) for k, v in splits.items()}
print({k: len(v) for k, v in splits.items()}, "| eval:", EVAL_SPLITS)

## 3. Merge the adapter into the base model (fp16)

In [ ]:
from peft import PeftModel
MERGED = f"{WORK}/models/merged_fp16"
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
if not os.path.exists(f"{MERGED}/config.json"):
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="cpu", low_cpu_mem_usage=True)
    merged = PeftModel.from_pretrained(base, ADAPTER_REPO).merge_and_unload()
    merged.save_pretrained(MERGED, safe_serialization=True); tok.save_pretrained(MERGED)
    del merged, base; gc.collect()
N_LAYERS = json.load(open(f"{MERGED}/config.json"))["num_hidden_layers"]
size_gb = sum(os.path.getsize(f) for f in glob.glob(f"{MERGED}/*.safetensors")) / 1e9
print(f"merged model: {N_LAYERS} layers, {size_gb:.2f} GB on disk")

## 4. Contrastive pairs: debit ↔ credit
The clean prompt is the exact chat prompt the model was trained on, followed by the start of its answer up to the `type` value. The corrupted prompt is the same SMS with the debit word swapped for its credit counterpart. Only pairs with identical token length and single-token answers are kept, as EAP requires.

In [ ]:
SWAPS = [("debited", "credited"), ("Debited", "Credited"), ("DEBITED", "CREDITED"), ("डेबिट", "क्रेडिट"),
         ("debit hue", "credit hue"), ("debit hua", "credit hua")]
PREFIX = '{"is_transaction": true, "type": "'
def chat_prompt(sms):
    return tok.apply_chat_template(build_messages(sms), tokenize=False, add_generation_prompt=True) + PREFIX
first = lambda s: tok(s, add_special_tokens=False)["input_ids"]
d_ids, c_ids = first("debit"), first("credit")
ANS_CLEAN, ANS_CORR = tok.decode(d_ids[:1]), tok.decode(c_ids[:1])
print("answer tokens:", d_ids, c_ids, "->", repr(ANS_CLEAN), repr(ANS_CORR))
pairs, rng = [], random.Random(SEED)
cands = [r for r in splits["train_v3"] if r["gold"]["is_transaction"] and r["gold"]["type"] == "debit"]
rng.shuffle(cands)
for r in cands:
    for a, b in SWAPS:
        if a in r["sms"]:
            cp, xp = chat_prompt(r["sms"]), chat_prompt(r["sms"].replace(a, b))
            if len(first(cp)) == len(first(xp)):
                pairs.append({"clean_prompt": cp, "clean_answer": ANS_CLEAN, "corrupt_prompt": xp, "corrupt_answer": ANS_CORR,
                              "id": r["id"], "lang": r["lang"]})
            break
    if len(pairs) >= N_PAIRS: break
write_jsonl(f"{WORK}/data/circuit_pairs.jsonl", pairs)
print(len(pairs), "aligned pairs | by language:", dict(Counter(p["lang"] for p in pairs)))
print("example corrupted SMS tail:", pairs[0]["corrupt_prompt"][-260:] if pairs else None)

## 5. CircuitKIT discovery (node-level EAP-IG, run in its own process)
Float16 on a T4 (bfloat16 is not native there), batch size 1, 3 IG steps, CircuitKIT's TransformerLens memory patch on. The result is one importance score per attention head and MLP block, summed per layer.

In [ ]:
DISCOVER = r"""
import os, sys, json
os.environ["CIRCUITKIT_TL_MEMORY_PATCH"] = "1"
import circuitkit as ck
from circuitkit import register_task
from circuitkit.data.normalized_task import NormalizedTaskSpec
from circuitkit.data.normalized import NormalizedDataset, ContrastiveRecord, DatasetShape
W, merged = sys.argv[1], sys.argv[2]
pairs = [json.loads(l) for l in open(f"{W}/data/circuit_pairs.jsonl", encoding="utf-8")]
recs = [ContrastiveRecord(record_id=str(i), clean_prompt=p["clean_prompt"], clean_answer=p["clean_answer"],
                          corrupt_prompt=p["corrupt_prompt"], corrupt_answer=p["corrupt_answer"]) for i, p in enumerate(pairs)]
ds = NormalizedDataset(name="sms_direction", shape=DatasetShape.QA, records=recs)
register_task(NormalizedTaskSpec(ds, name="sms_direction", templated=True))
model = ck.load_model(merged, dtype="float16")
circuit = ck.discover(model, "sms_direction", algorithm="eap-ig", level="node", n_examples=len(pairs),
                      batch_size=1, output_path=f"{W}/results/circuit_sms_direction.pt")
json.dump({"scores": circuit.scores, "top": circuit.top_nodes(20)}, open(f"{W}/results/circuit_scores.json", "w"), indent=1)
print("top nodes:", circuit.top_nodes(10))
"""
open(f"{WORK}/discover.py", "w").write(DISCOVER)
LAYER_SCORES, PROTECT, CK_OK = None, [], False
if RUN_CIRCUITKIT and OK_CK and pairs:
    t0 = time.time()
    env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    proc = subprocess.run([sys.executable, f"{WORK}/discover.py", WORK, MERGED], capture_output=True, text=True, env=env)
    print(proc.stdout[-3000:]); print(proc.stderr[-3000:])
    if proc.returncode == 0 and os.path.exists(f"{WORK}/results/circuit_scores.json"):
        sc = json.load(open(f"{WORK}/results/circuit_scores.json"))["scores"]
        LAYER_SCORES = [0.0] * N_LAYERS
        for node, v in sc.items():
            m = re.match(r"A(\d+)\.\d+", node) or re.match(r"MLP (\d+)", node)
            if m and int(m.group(1)) < N_LAYERS:
                LAYER_SCORES[int(m.group(1))] += abs(float(v))
        PROTECT = sorted(sorted(range(N_LAYERS), key=lambda i: -LAYER_SCORES[i])[:N_PROTECT])
        CK_OK = True
        print(f"CircuitKIT done in {(time.time() - t0) / 60:.1f} min | protected layers (top {N_PROTECT}):", PROTECT)
    else:
        print("⚠️ CircuitKIT discovery failed; circuit-protected versions are skipped (uniform versions still run)")
RANDOM_PROTECT = sorted(random.Random(SEED).sample([i for i in range(N_LAYERS) if i not in PROTECT] or list(range(N_LAYERS)), N_PROTECT))
print("random-protection control:", RANDOM_PROTECT)
json.dump({"layer_scores": LAYER_SCORES, "protect": PROTECT, "random_protect": RANDOM_PROTECT, "n_pairs": len(pairs),
           "circuitkit_ok": CK_OK}, open(f"{WORK}/results/protection.json", "w"), indent=1)

## 6. GPU versions (bitsandbytes) and their evaluation

In [ ]:
def generate(model, prompts, batch=GEN_BATCH, max_new=MAX_NEW):
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    out = [None] * len(prompts)
    for k in range(0, len(order), batch):
        idx = order[k:k + batch]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            g = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        for j, i in enumerate(idx):
            out[i] = tok.decode(g[j, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return out

RESULTS = {}
def evaluate(name, model, extra=None):
    torch.cuda.reset_peak_memory_stats()
    row = {"name": name, **(extra or {})}
    try: row["weights_gb"] = round(model.get_memory_footprint() / 1e9, 2)
    except Exception: pass
    t0, n = time.time(), 0
    for sp in EVAL_SPLITS:
        rows = splits[sp]
        outs = generate(model, [tok.apply_chat_template(build_messages(r["sms"]), tokenize=False, add_generation_prompt=True) for r in rows])
        n += len(rows)
        sc = [S.score_example(o, r["gold"]) for o, r in zip(outs, rows)]
        m = S.aggregate(sc)
        ci = S.bootstrap_ci([float(x["exact_match"]) for x in sc], seed=SEED)
        row[sp] = {"em": m["exact_match"], "ci": ci, "halluc": m.get("hallucination_rate"), "schema": m["schema_valid"]}
        write_jsonl(f"{WORK}/results/predictions_{name}_{sp}.jsonl",
                    [{"id": r["id"], "output": o, "exact_match": s["exact_match"]} for r, o, s in zip(rows, outs, sc)])
    row["peak_gpu_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    row["sms_per_s"] = round(n / (time.time() - t0), 2)
    RESULTS[name] = row
    print(name, "| weights", row.get("weights_gb"), "GB | peak", row["peak_gpu_gb"], "GB |",
          " ".join(f"{sp} EM {100 * row[sp]['em']:.1f}" for sp in EVAL_SPLITS), "|", row["sms_per_s"], "SMS/s")
    json.dump(RESULTS, open(f"{WORK}/results/quant_results.json", "w"), indent=1, default=str)

def skip_layers(layers):
    return ["lm_head"] + [f"model.layers.{i}" for i in layers]

VARIANTS = [("fp16", None, {"precision": "fp16"}),
            ("int8", BitsAndBytesConfig(load_in_8bit=True), {"precision": "int8 (LLM.int8)"}),
            ("nf4_uniform", BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16), {"precision": "nf4, all layers"})]
if CK_OK:
    for tag, layers in [("nf4_circuit", PROTECT), ("nf4_random", RANDOM_PROTECT)]:
        VARIANTS.append((tag, BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                                                 llm_int8_skip_modules=skip_layers(layers)),
                         {"precision": f"nf4, layers {layers} kept fp16"}))
for name, qcfg, extra in VARIANTS:
    try:
        kw = {"quantization_config": qcfg} if qcfg else {"torch_dtype": torch.float16}
        model = AutoModelForCausalLM.from_pretrained(MERGED, device_map={"": 0}, **kw).eval()
        evaluate(name, model, extra)
    except Exception:
        traceback.print_exc(); print(f"⚠️ {name} failed; continuing")
    finally:
        try: del model
        except NameError: pass
        gc.collect(); torch.cuda.empty_cache()

## 7. GGUF versions for CPU and the 2 GB tier (llama.cpp)
llama.cpp quantization also compresses the 262k-token embedding table (about 1 GB in fp16), which is what makes a 2 GB version possible. Circuit-protected versions keep every tensor of the protected layers at Q8_0 via `--tensor-type`. Evaluated on CPU (Kaggle's 4 vCPUs) with `llama-server`.

In [ ]:
GGUF = {}
if RUN_GGUF:
    try:
        LC = f"{WORK}/llama.cpp"
        if not os.path.isdir(LC):
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LC], check=True)
        t0 = time.time()
        subprocess.run(["cmake", "-B", f"{LC}/build", "-S", LC, "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"],
                       check=True, capture_output=True)
        subprocess.run(["cmake", "--build", f"{LC}/build", "-j", "4", "--target", "llama-quantize", "llama-server"],
                       check=True, capture_output=True)
        print(f"llama.cpp built in {(time.time() - t0) / 60:.1f} min")
        F16 = f"{WORK}/models/sms-f16.gguf"
        r = subprocess.run([sys.executable, f"{LC}/convert_hf_to_gguf.py", MERGED, "--outfile", F16, "--outtype", "f16"],
                           capture_output=True, text=True)
        print(r.stderr[-800:])
        def quant(name, qtype, protect=None):
            out = f"{WORK}/models/tiny-aya-sms-{name}.gguf"
            cmd = [f"{LC}/build/bin/llama-quantize"]
            if protect:
                cmd += ["--tensor-type", "blk\\.(" + "|".join(map(str, protect)) + ")\\..*=q8_0"]
            r = subprocess.run(cmd + [F16, out, qtype], capture_output=True, text=True)
            if r.returncode or not os.path.exists(out):
                print(name, "quantize failed:", r.stderr[-600:]); return
            GGUF[name] = {"path": out, "gb": round(os.path.getsize(out) / 1e9, 2), "type": qtype, "protect": protect or []}
            print(name, GGUF[name]["gb"], "GB")
        quant("q8_0", "Q8_0")
        quant("q4_k_m", "Q4_K_M")
        quant("q3_k_m", "Q3_K_M")
        if CK_OK:
            quant("q4_k_m-circuit", "Q4_K_M", PROTECT)
            quant("q3_k_m-circuit", "Q3_K_M", PROTECT)
            quant("q3_k_m-random", "Q3_K_M", RANDOM_PROTECT)
    except Exception:
        traceback.print_exc(); print("⚠️ GGUF step failed")

In [ ]:
import requests
def eval_gguf(name, info, rows):
    port = 8099
    srv = subprocess.Popen([f"{WORK}/llama.cpp/build/bin/llama-server", "-m", info["path"], "--port", str(port),
                            "-c", "2048", "-t", str(os.cpu_count() or 4), "--no-webui"],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    try:
        for _ in range(120):
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200: break
            except Exception: pass
            time.sleep(2)
        outs, t0 = [], time.time()
        for r in rows:
            prompt = tok.apply_chat_template(build_messages(r["sms"]), tokenize=False, add_generation_prompt=True)
            js = requests.post(f"http://127.0.0.1:{port}/completion", json={"prompt": prompt, "n_predict": MAX_NEW,
                               "temperature": 0, "cache_prompt": True}, timeout=600).json()
            outs.append(js.get("content", ""))
        sec = (time.time() - t0) / max(len(rows), 1)
    finally:
        srv.terminate(); srv.wait()
    sc = [S.score_example(o, r["gold"]) for o, r in zip(outs, rows)]
    m = S.aggregate(sc)
    RESULTS[f"gguf_{name}"] = {"name": f"gguf_{name}", "precision": info["type"] + (f", layers {info['protect']} kept Q8_0" if info["protect"] else ""),
                               "file_gb": info["gb"], "eval_rows": len(rows), "em": m["exact_match"],
                               "ci": S.bootstrap_ci([float(x["exact_match"]) for x in sc], seed=SEED),
                               "halluc": m.get("hallucination_rate"), "cpu_sec_per_sms": round(sec, 1)}
    print(f"gguf {name}: {info['gb']} GB | EM {100 * m['exact_match']:.1f} on {len(rows)} SMS | {sec:.1f} s/SMS on CPU")
    json.dump(RESULTS, open(f"{WORK}/results/quant_results.json", "w"), indent=1, default=str)

GGUF_ROWS = (splits.get("challenge") or splits[EVAL_SPLITS[0]])[:GGUF_EVAL_N]
for name, info in GGUF.items():
    try: eval_gguf(name, info, GGUF_ROWS)
    except Exception: traceback.print_exc()

## 8. Results: size vs accuracy

In [ ]:
def pct(x): return "–" if x is None or (isinstance(x, float) and x != x) else f"{100 * x:.1f}"
lines = ["| Version | Precision | Size (GB) | Fits GPU | " + " | ".join(f"{sp} EM [95% CI]" for sp in EVAL_SPLITS) + " | SMS/s (T4) |",
         "|---|---|---|---|" + "---|" * len(EVAL_SPLITS) + "---|"]
def tier(gb):
    return "–" if gb is None else ("2 GB" if gb + 0.3 <= 2 else "4 GB" if gb + 0.4 <= 4 else "8 GB" if gb + 0.6 <= 8 else "16 GB")
for k, v in RESULTS.items():
    if k.startswith("gguf_"): continue
    gb = v.get("peak_gpu_gb")
    lines.append(f"| {k} | {v.get('precision')} | {v.get('weights_gb')} (peak {gb}) | {tier(gb)} | "
                 + " | ".join(f"{pct(v[sp]['em'])} [{pct(v[sp]['ci'][0])}–{pct(v[sp]['ci'][1])}]" for sp in EVAL_SPLITS) + f" | {v.get('sms_per_s')} |")
GPU_TABLE = "\n".join(lines)
gl = ["| GGUF version | Precision | File (GB) | Fits | EM on hand-written (n) [95% CI] | CPU s/SMS |", "|---|---|---|---|---|---|"]
for k, v in RESULTS.items():
    if not k.startswith("gguf_"): continue
    gl.append(f"| {k[5:]} | {v['precision']} | {v['file_gb']} | {tier(v['file_gb'])} | {pct(v['em'])} ({v['eval_rows']}) [{pct(v['ci'][0])}–{pct(v['ci'][1])}] | {v['cpu_sec_per_sms']} |")
GGUF_TABLE = "\n".join(gl)
print(GPU_TABLE + "\n\n" + GGUF_TABLE)
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4))
    sp0 = "challenge" if "challenge" in EVAL_SPLITS else EVAL_SPLITS[0]
    for k, v in RESULTS.items():
        x = v.get("file_gb") if k.startswith("gguf_") else v.get("weights_gb")
        y = v.get("em") if k.startswith("gguf_") else v[sp0]["em"]
        mk = "s" if "circuit" in k else ("x" if "random" in k else "o")
        ax.scatter(x, 100 * y, marker=mk); ax.annotate(k.replace("gguf_", ""), (x, 100 * y), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel("model size (GB)"); ax.set_ylabel(f"exact match, {sp0} (%)")
    ax.set_title("Size vs accuracy (square = circuit-protected, x = random-protected)")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); fig.savefig(f"{WORK}/figures/size_vs_accuracy.png", dpi=160)
except Exception:
    traceback.print_exc()
open(f"{WORK}/results/quant_tables.md", "w").write("## GPU versions\n" + GPU_TABLE + "\n\n## GGUF versions\n" + GGUF_TABLE)

## 9. Publish the quantized versions (4th model repo)

In [ ]:
prot = json.load(open(f"{WORK}/results/protection.json"))
CARD = f"""---
base_model: {BASE_MODEL}
license: cc-by-nc-4.0
language: [en, hi]
library_name: gguf
tags: [sms, upi, banking, information-extraction, structured-output, hinglish, quantized, gguf, circuitkit]
---
# Tiny Aya SMS Extract · quantized versions for 2 / 4 / 8 / 16 GB

Smaller versions of the fine-tuned Tiny Aya Fire SMS extractor (adapter: `{ADAPTER_REPO}`). Author: Anshul J Amin.

**Circuit-aware:** CircuitKIT (node-level EAP-IG on {prot['n_pairs']} debit↔credit contrastive pairs) ranked the layers by their role in the debit-vs-credit decision.
Protected layers (kept at higher precision): `{prot['protect']}`. Random-protection control: `{prot['random_protect']}`.

## GGUF files (llama.cpp, CPU or GPU)
{GGUF_TABLE}

## GPU versions (transformers + bitsandbytes, built at load time from the merged model)
{GPU_TABLE}

Load the circuit-protected 4-bit version:
```python
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         llm_int8_skip_modules=["lm_head"] + [f"model.layers.{{i}}" for i in {prot['protect']}])
# merge the adapter into the base first (see the GitHub README), then from_pretrained(merged_dir, quantization_config=cfg)
```
Prompt format: see `src/smsgen.py` (`build_messages`) in https://github.com/Anshul-Amin/tiny-aya-sms-extract.
Licence: CC-BY-NC 4.0 (inherits Tiny Aya). Data are synthetic; bank names are fictional.
"""
open(f"{WORK}/README_quant.md", "w").write(CARD)
if PUSH_TO_HUB and not QUICK_MODE:
    api.create_repo(QUANT_REPO, repo_type="model", exist_ok=True)
    api.upload_file(path_or_fileobj=f"{WORK}/README_quant.md", path_in_repo="README.md", repo_id=QUANT_REPO)
    for name, info in GGUF.items():
        api.upload_file(path_or_fileobj=info["path"], path_in_repo=os.path.basename(info["path"]), repo_id=QUANT_REPO)
        print("uploaded", os.path.basename(info["path"]))
    api.upload_folder(folder_path=f"{WORK}/results", path_in_repo="results", repo_id=QUANT_REPO)
    api.upload_folder(folder_path=f"{WORK}/figures", path_in_repo="figures", repo_id=QUANT_REPO)
    try: api.add_collection_item(MODEL_COLLECTION, item_id=QUANT_REPO, item_type="model", exists_ok=True)
    except Exception as e: print("collection:", e)
    print("https://huggingface.co/" + QUANT_REPO)
else:
    print("QUICK_MODE: not uploaded")